# AAPL 真實公開資料 E0–E4 實驗
450 篇資料的本地首輪已完成；這個筆記本可重新收集並執行相同的固定規格。
使用 CPU 與本地模型，無須付費 API key。免費 Colab 的資源和執行時間有限。
這是回溯探索，尚未有獨立人工事件答案，也不能排除歷史文章修改。
原四篇合成新聞的標註流程與本實驗是分開的。


## 1. 取得研究分支
新 runtime 只 clone 一次；重啟後只需重新切換資料夾。

In [ ]:
!git clone --branch research/event-extraction-e0-e4 --single-branch https://github.com/yoyocar2333/stockllm-pit-news.git /content/stockllm-pit-news
%cd /content/stockllm-pit-news


## 2. 安裝與模型
若提示重啟，重啟後執行 `%cd /content/stockllm-pit-news` 再繼續。

In [ ]:
%pip install -r requirements-research.txt
!python -m spacy download en_core_web_sm


## 3. 收集真實資料
抓取 Apple 英文 Newsroom 2023–2025 文章與 AAPL 日行情；有缓存的網頁會重用。
不要把原始新聞全文或模型權重上傳到公開 GitHub。


In [ ]:
!python -m experiments.collect_apple_newsroom


## 4. 本地情緒與事件抽取
全文所有 token 分段處理。首次需要下載 FinBERT；中斷後已完成的段可重用。
輸出資料夾既有輸入／模型雜湊不同時程式會拒絕混用，請保留本輪或另外建立版本。


In [ ]:
import os
os.environ['HF_HOME'] = '/content/stockllm-pit-news/outputs/model-cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
!python -m experiments.infer_apple_public --threads 2 --batch-size 8


## 5. 固定時間切分的預測比較與報告
2023 訓練、2024 驗證、2025 測試；不根據測試分數重新調參。
再次抓取可能遇到網站文章修改，對照紀錄中的輸入雜湊再解讀結果。


In [ ]:
!python -m experiments.evaluate_apple_public
!python -m experiments.report_apple_public
!python -m pytest -q


In [ ]:
import pandas as pd
from IPython.display import Image, display
metrics = pd.read_csv('outputs/apple_public_v1/metrics.csv')
display(metrics[metrics['split']=='test'])
display(Image(filename='outputs/apple_public_v1/comparison.png'))


## 6. 下載完整衍生結果
內含指標、逐日預測、特徵、來源雜湊與報告；不打包新聞全文／HTML 或權重。
這輪還不能宣稱事件抽取準確率，之後需要固定樣本的獨立盲標。


In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/aapl_public_e0_e4_results','zip','outputs/apple_public_v1')
files.download('/content/aapl_public_e0_e4_results.zip')
